# Single Participant Occipital QC + ERP Waveform Check

This piplein is for inspecting one participant each before loopin all in another script.

Steps:
1. Set dataset, path, subject id
2. Load EDF.
3. Set channel types dan montage.
4. Extract P3b events dari `MarkerValueInt`.
5. Filter data: notch 50 Hz nad bandpass 0.1-30 Hz.
6. Plot raw filtered EEG untuk manual bad-channel inspection.
7. input bad channels manually.
8. Optional: interpolate bad channels.
9. Apply average reference.
10. Epoch target and non-target.
11. Clean epochs dengan AutoReject.
12. Plot waveform O1 and O2 for that

Catatan:
- Karena analisis fokus pada O1/O2, inspect O1 dan O2 dengan ekstra hati-hati.
- Kalau O1 atau O2 bad berat, catat untuk exclude region tersebut di manual QC CSV.
- Kalau channel lain bad, bisa dicatat sebagai bad channel dan diinterpolate sebelum average reference.

In [ ]:
from pathlib import Path

import mne
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from autoreject import AutoReject


# =========================
# EDIT THIS PART
# =========================

DATASET_NAME = "ND"  # "UG", "GSAL", or "ND"

BIDS_ROOT = Path(
    "/Users/miftahfaizah/Library/CloudStorage/OneDrive-UniversityofLeeds/"
    "PHD JOURNEY/YBMAP/CN_DATASET/Notre_Dame_2026/Bids_conversion"
)

# Example EXCLUDED ND_26 participants (change SUBJECT to inspect why each was excluded):
#   sub-ST205EQ  -> O1;O2 bad  (both occipital bad -> excluded; analysis needs O1/O2)
#   sub-AM712RT  -> O1;O2 bad
#   sub-LO103ND  -> P7;P8;T7;T8 bad
#   sub-LA502LS  -> 7 channels bad (AF4;F4;F8;FC6;O2;P8;T8) - too noisy overall
#   sub-KA205TN  -> 6 channels bad (AF3;F7;FC5;O1;P7;T7)
SUBJECT = "sub-AM712RT"
TASK = "p3b"

ANALYSIS_BASE = Path(
    "/Users/miftahfaizah/Library/CloudStorage/OneDrive-UniversityofLeeds/"
    "PHD JOURNEY/YBMAP/CN_DATASET/CN_analysis/YBMAP_P3b_review/2_Preprocessing/ND_26"
)

MANUAL_BAD_CHANNELS = []  # example: ["F7", "P8"]
INTERPOLATE_BADS = True

# =========================
# SETTINGS
# =========================

EEG_CHANNELS = [
    "AF3", "F7", "F3", "FC5", "T7", "P7", "O1",
    "O2", "P8", "T8", "FC6", "F4", "F8", "AF4",
]

REGIONS = {
    "O1": ["O1"],
    "O2": ["O2"],
    "P7": ["P7"],
    "P8": ["P8"],
}

EVENT_ID = {
    "target": 111,
    "non_target": 222,
}

P3B_WINDOW = (0.3, 0.6)

TMIN = -0.2
TMAX = 0.8
BASELINE = (None, 0)

EXPECTED_TARGET = 40
EXPECTED_NON_TARGET = 160
REQUIRE_EXPECTED_COUNTS = True

RANDOM_STATE = 42


# =========================
# HELPER FUNCTIONS
# =========================

def set_channel_types(raw):
    channel_types = {}

    for ch in raw.ch_names:
        if ch in EEG_CHANNELS:
            channel_types[ch] = "eeg"
        elif ch == "MarkerValueInt":
            channel_types[ch] = "stim"
        else:
            channel_types[ch] = "misc"

    raw.set_channel_types(channel_types)
    return raw


def extract_events_from_marker(raw, marker_channel="MarkerValueInt", marker_scale=1e6, min_gap_s=0.05):
    marker_data = raw.copy().pick(marker_channel).get_data()[0]
    marker_scaled = np.round(marker_data * marker_scale).astype(int)

    sfreq = raw.info["sfreq"]
    nonzero_idx = np.where(marker_scaled != 0)[0]

    event_samples = []
    event_codes = []

    for idx in nonzero_idx:
        if len(event_samples) == 0:
            event_samples.append(idx)
            event_codes.append(marker_scaled[idx])
        else:
            previous_idx = event_samples[-1]
            previous_code = event_codes[-1]

            is_new_code = marker_scaled[idx] != previous_code
            is_far_enough = (idx - previous_idx) / sfreq > min_gap_s

            if is_new_code or is_far_enough:
                event_samples.append(idx)
                event_codes.append(marker_scaled[idx])

    return np.column_stack([
        event_samples,
        np.zeros(len(event_samples), dtype=int),
        event_codes,
    ]).astype(int)


def select_p3b_stim_events(events_all, require_expected_counts=REQUIRE_EXPECTED_COUNTS):
    p3b_bounds = events_all[events_all[:, 2] == 5]

    if len(p3b_bounds) < 2:
        raise ValueError(f"Expected at least two P3b task markers coded 5, found {len(p3b_bounds)}")

    p3b_start_sample = p3b_bounds[0, 0]
    p3b_end_sample = p3b_bounds[1, 0]

    events = events_all[
        (events_all[:, 0] > p3b_start_sample)
        & (events_all[:, 0] < p3b_end_sample)
        & (np.isin(events_all[:, 2], [EVENT_ID["target"], EVENT_ID["non_target"]]))
    ].copy()

    counts = pd.Series(events[:, 2]).value_counts().to_dict()
    n_target = counts.get(EVENT_ID["target"], 0)
    n_non_target = counts.get(EVENT_ID["non_target"], 0)

    if require_expected_counts and (n_target != EXPECTED_TARGET or n_non_target != EXPECTED_NON_TARGET):
        raise ValueError(
            "Unexpected P3b stimulus counts after task-boundary filtering: "
            f"111={n_target}, 222={n_non_target}, total={len(events)}"
        )

    return events, p3b_start_sample, p3b_end_sample, n_target, n_non_target


# =========================
# LOAD DATA
# =========================

eeg_dir = BIDS_ROOT / SUBJECT / "eeg"
eeg_file = eeg_dir / f"{SUBJECT}_task-{TASK}_eeg.edf"
events_file = eeg_dir / f"{SUBJECT}_task-{TASK}_events.tsv"

print("Dataset:", DATASET_NAME)
print("Subject:", SUBJECT)
print("EDF exists:", eeg_file.exists())
print("Events TSV exists:", events_file.exists())

raw = mne.io.read_raw_edf(eeg_file, preload=True, verbose=False)
raw = set_channel_types(raw)

montage = mne.channels.make_standard_montage("standard_1020")
raw.set_montage(montage, match_case=False, on_missing="ignore")

events_all = extract_events_from_marker(raw)
events, p3b_start_sample, p3b_end_sample, n_target_detected, n_nontarget_detected = select_p3b_stim_events(events_all)

print("All marker events:", len(events_all))
print("P3b events used:", len(events))
print("Target detected:", n_target_detected)
print("Non-target detected:", n_nontarget_detected)


# =========================
# FILTER FULL RAW
# =========================

raw_filt = raw.copy()
raw_filt.notch_filter(freqs=50, picks="eeg", verbose=False)
raw_filt.filter(l_freq=0.1, h_freq=30, picks="eeg", verbose=False)


# =========================
# CROP TO P3B SEGMENT FOR INSPECTION
# =========================

sfreq = raw_filt.info["sfreq"]

p3b_tmin = max((p3b_start_sample / sfreq) - 2, 0)
p3b_tmax = (p3b_end_sample / sfreq) + 2

raw_p3b = raw_filt.copy().crop(
    tmin=p3b_tmin,
    tmax=p3b_tmax,
)

print("P3b segment start time:", p3b_tmin)
print("P3b segment end time:", p3b_tmax)
print("P3b segment duration:", p3b_tmax - p3b_tmin)

raw_p3b.plot(
    picks="eeg",
    duration=10,
    n_channels=14,
    scalings=dict(eeg=100e-6),
    title=f"{DATASET_NAME} {SUBJECT}: P3b segment bad-channel inspection",
)

In [ ]:
EXCLUDE_O1 = False 
EXCLUDE_O2 = False
EXCLUDE_P7 = False
EXCLUDE_P8 = False
EXCLUDE_SUBJECT = False
QC_NOTES = "All look ok"

In [ ]:
# =========================
# APPLY BAD CHANNELS + INTERPOLATE + REFERENCE
# =========================

clicked_bad_channels = [str(ch) for ch in raw_p3b.info["bads"]]
manual_bad_channels = sorted(set(clicked_bad_channels + MANUAL_BAD_CHANNELS))

valid_bad_channels = [
    ch for ch in manual_bad_channels
    if ch in EEG_CHANNELS
]

invalid_bad_channels = [
    ch for ch in manual_bad_channels
    if ch not in EEG_CHANNELS
]

raw_filt.info["bads"] = valid_bad_channels

print("Clicked bad channels from P3b plot:", clicked_bad_channels)
print("Manual typed bad channels:", MANUAL_BAD_CHANNELS)
print("Valid bad channels used:", valid_bad_channels)
print("Invalid bad channels ignored:", invalid_bad_channels)

if INTERPOLATE_BADS and len(raw_filt.info["bads"]) > 0:
    raw_clean_base = raw_filt.copy().interpolate_bads(
        reset_bads=False,
        verbose=False,
    )
else:
    raw_clean_base = raw_filt.copy()

print("Bad channels before interpolation:", raw_filt.info["bads"])
print("Bad channels after interpolation:", raw_clean_base.info["bads"])
print("Interpolation used:", INTERPOLATE_BADS and len(raw_filt.info["bads"]) > 0)

raw_ref = raw_clean_base.copy().set_eeg_reference(
    ref_channels="average",
    projection=False,
    verbose=False,
)


# =========================
# EPOCH + AUTOREJECT
# =========================

epochs = mne.Epochs(
    raw_ref,
    events,
    event_id=EVENT_ID,
    tmin=TMIN,
    tmax=TMAX,
    baseline=BASELINE,
    picks="eeg",
    preload=True,
    reject=None,
    verbose=False,
)

ar = AutoReject(
    n_interpolate=[1, 2, 4],
    consensus=np.linspace(0.2, 0.8, 4),
    random_state=RANDOM_STATE,
    verbose=False,
)

epochs_clean, reject_log = ar.fit_transform(
    epochs,
    return_log=True,
)

n_original = len(epochs)
n_clean = len(epochs_clean)
n_dropped = int(reject_log.bad_epochs.sum())
retained_prop = n_clean / n_original if n_original > 0 else np.nan

n_target_clean = len(epochs_clean["target"])
n_nontarget_clean = len(epochs_clean["non_target"])

print("Original epochs:", n_original)
print("Clean epochs:", n_clean)
print("Dropped epochs:", n_dropped)
print("Retained proportion:", retained_prop)
print("Clean target epochs:", n_target_clean)
print("Clean non-target epochs:", n_nontarget_clean)


# =========================
# PLOT + SAVE O1/O2 WAVEFORMS
# =========================

waveform_dir = ANALYSIS_BASE / "outputs" / "individual_waveforms_nd26"
waveform_dir.mkdir(parents=True, exist_ok=True)

waveform_rows_single = []

fig, axes = plt.subplots(
    1,
    len(REGIONS),
    figsize=(4 * len(REGIONS), 4),
    sharex=True,
    sharey=True,
    squeeze=False,
)

for col_idx, (region_name, channels) in enumerate(REGIONS.items()):
    ax = axes[0, col_idx]

    target_evoked = epochs_clean["target"].copy().pick(channels).average()
    nontarget_evoked = epochs_clean["non_target"].copy().pick(channels).average()

    times_ms = target_evoked.times * 1000
    target_uv = target_evoked.data.mean(axis=0) * 1e6
    nontarget_uv = nontarget_evoked.data.mean(axis=0) * 1e6
    difference_uv = target_uv - nontarget_uv

    ax.plot(times_ms, target_uv, label="Target", color="tab:blue")
    ax.plot(times_ms, nontarget_uv, label="Non-target", color="tab:orange")
    ax.plot(times_ms, difference_uv, label="Target - Non-target", color="tab:green", linestyle="--")

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvspan(300, 600, color="grey", alpha=0.2)

    ax.set_title(f"{DATASET_NAME} {SUBJECT} {region_name}")
    ax.set_xlabel("Time (ms)")

    if col_idx == 0:
        ax.set_ylabel("Amplitude (uV)")

    ax.legend()

    for time_ms, target_amp, nontarget_amp, diff_amp in zip(
        times_ms,
        target_uv,
        nontarget_uv,
        difference_uv,
    ):
        waveform_rows_single.append({
            "dataset": DATASET_NAME,
            "subject": SUBJECT,
            "region": region_name,
            "time_ms": time_ms,
            "target_uv": target_amp,
            "non_target_uv": nontarget_amp,
            "target_minus_non_target_uv": diff_amp,
            "manual_bad_channels": ";".join(raw_filt.info["bads"]),
            "interpolate_bads": INTERPOLATE_BADS,
            "n_epochs_original": n_original,
            "n_epochs_clean": n_clean,
            "n_epochs_dropped": n_dropped,
            "retained_prop": retained_prop,
            "n_target_clean": n_target_clean,
            "n_non_target_clean": n_nontarget_clean,
        })

plt.suptitle(f"Single Participant Occipital P3b: {DATASET_NAME} {SUBJECT}")
plt.tight_layout()

waveform_fig_out = waveform_dir / f"{DATASET_NAME}_{SUBJECT}_occipital_waveform.png"
waveform_csv_out = waveform_dir / f"{DATASET_NAME}_{SUBJECT}_occipital_waveform.csv"

plt.savefig(
    waveform_fig_out,
    dpi=300,
    bbox_inches="tight",
)

waveform_single = pd.DataFrame(waveform_rows_single)
waveform_single.to_csv(waveform_csv_out, index=False)

print("Saved waveform figure to:", waveform_fig_out)
print("Saved waveform CSV to:", waveform_csv_out)

plt.show()


# =========================
# SAVE MANUAL QC DECISION
# =========================

qc_out = ANALYSIS_BASE / "manual_occipital_qc_ug_trainingday.csv"

qc_row = pd.DataFrame([{
    "dataset": DATASET_NAME,
    "subject": SUBJECT,
    "bad_channels": ";".join(raw_filt.info["bads"]),
    "exclude_O1": EXCLUDE_O1,
    "exclude_O2": EXCLUDE_O2,
    "exclude_P7": EXCLUDE_P7,
    "exclude_P8": EXCLUDE_P8,
    "exclude_subject": EXCLUDE_SUBJECT,
    "notes": QC_NOTES,
}])

if qc_out.exists():
    qc_existing = pd.read_csv(qc_out)

    qc_existing = qc_existing[
        ~(
            (qc_existing["dataset"] == DATASET_NAME)
            & (qc_existing["subject"] == SUBJECT)
        )
    ]

    qc_all = pd.concat([qc_existing, qc_row], ignore_index=True)
else:
    qc_all = qc_row

qc_all.to_csv(qc_out, index=False)

print("Saved manual QC to:", qc_out)
qc_all.tail()